# DEPI基金绩效归因分析 - 研报复现
**研报**：DEA模型基于净值构建DEPI指标，横向比较同类基金（华泰金工，2020-08-21）

---

In [ ]:
# 导入所有模块
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import config
from source.data_loader import (
    get_fund_list, get_fund_nav_history,
    get_fund_fee_rate, get_benchmark_history,
)
from source.factor import build_factor_table
from source.backtest import DEPIEngine, backtest_depi
from source.plot import (
    setup_chinese_font,
    plot_depi_distribution,
    plot_depi_timeseries,
    plot_depi_bar_topn,
)
from source.utils import save_df

print('模块加载完成')

## Step 1: 数据获取

In [ ]:
# 获取基金列表
fund_list = get_fund_list(n=config.SAMPLE_SIZE)
fund_list.head()

In [ ]:
# 获取基金净值历史
nav_dict = get_fund_nav_history(
    fund_list['基金代码'].tolist(),
    config.BACKTEST_START, config.BACKTEST_END
)
print(f'有效基金: {len(nav_dict)}')

In [ ]:
# 获取费率
fee_df = get_fund_fee_rate(fund_list['基金代码'].tolist())
fee_df.head()

In [ ]:
# 获取沪深300基准
benchmark_df = get_benchmark_history(config.BENCHMARK_CODE)
benchmark_returns = benchmark_df['日收益率']
print(f'基准数据: {len(benchmark_df)}条')

## Step 2: 构建因子表

In [ ]:
factor_df = build_factor_table(
    nav_dict, benchmark_returns, fee_df, config.RISK_FREE_RATE
)
factor_df.describe()

## Step 3: DEPI截面分析

In [ ]:
engine = DEPIEngine()
depi_result = engine.fit_transform(
    factor_df,
    output_col='超额收益R',
    input_cols=config.INPUT_INDICATORS
)
print('统计摘要:', engine.get_summary())
depi_result[['基金代码','DEPI','DEPI_Rank','超额收益R',
             'volatility','fee_rate','timing_alpha','timing_beta']].head(15)

## Step 4: 可视化

In [ ]:
setup_chinese_font()
plot_depi_distribution(depi_result)

In [ ]:
plot_depi_bar_topn(depi_result, top_n=15)

## Step 5: 滚动回测

In [ ]:
depi_ts = backtest_depi(
    nav_dict, benchmark_returns, fee_df,
    start=config.BACKTEST_START,
    end=config.BACKTEST_END,
    freq=config.REBALANCE_FREQ,
    input_cols=config.INPUT_INDICATORS,
)
if not depi_ts.empty:
    plot_depi_timeseries(depi_ts, top_n=5)